# Etapa 01 — Filtrar as imagens com macrófagos infectados

- Autor: Clésio Gonçalves
- Criado em: 21 jun. 2026 · reorganizado em 17 nov. 2026

**Objetivo:** separar, entre todas as imagens anotadas no CVAT, apenas as que têm ao menos uma anotação `inside-macrophage`, ou seja, as que têm algum macrófago infectado.

| | |
|---|---|
| **Entrada** | `00_dataset_cvat/` (zips exportados do CVAT no formato YOLO 1.1; são extraídos automaticamente para `leish_N/obj_train_data/`) |
| **Saída** | `02_datasets_derivados/01_imagens_macrofagos_infectados/<lamina>/` (imagens inteiras + `.txt` originais + `obj.names`) |
| **Próxima etapa** | `02_gerar_recortes_512.ipynb` |

**Classes do `obj.names`:** `0 Non-macrophages`, `1 macrophage`, `2 leishmania` (alvo: fora do macrófago) e `3 inside-macrophage`.

**Resultado:** 599 imagens selecionadas de 609.

| Lâmina | leish_1 | leish_2 | leish_3 | leish_4 | leish_5 | leish_7 | leish_8 | leish_9 |
|---|---|---|---|---|---|---|---|---|
| Selecionadas / total | 67/67 | 46/52 | 11/14 | 47/47 | 127/127 | 95/95 | 134/134 | 72/73 |

As células são idempotentes: se a pasta de saída já tiver conteúdo, a etapa é pulada.

# Importações e parâmetros gerais

In [2]:
import os
import pandas as pd
import zipfile
import shutil
from pathlib import Path

In [3]:
# Diretório raiz do projeto
diretorio_raiz = Path("/mnt/arquivos/Doutorado")

# Entrada: dataset exportado do CVAT (zips no formato YOLO 1.1)
diretorio_cvat = os.path.join(diretorio_raiz, "00_dataset_cvat")

# Saída: imagens inteiras que contêm macrófago infectado
diretorio_saida = os.path.join(diretorio_raiz, "02_datasets_derivados", "01_imagens_macrofagos_infectados")

# extensão das imagens
extensao = '.png'

# subpasta com imagens e .txt dentro de cada lâmina extraída
subpasta_objetos = 'obj_train_data'

# classe usada para identificar imagens com macrófago infectado
classe_alvo = 'inside-macrophage'

## 1. Extração dos zips do CVAT

In [4]:
def descompacta_arquivos(diretorio_cvat):
    for index, arquivo in enumerate(os.listdir(diretorio_cvat)):
        if arquivo.endswith('.zip'):
            caminho_arquivo_zip = os.path.join(diretorio_cvat, arquivo)
            
            # Padroniza a pasta de destino
            partes = arquivo.split("_")
            nome_leish = "_".join(partes[1:3])
            caminho_pasta_destino = os.path.join(diretorio_cvat, nome_leish)
    
            # Extrair arquivos zip
            with zipfile.ZipFile(caminho_arquivo_zip,"r") as arquivo_zip:
                arquivo_zip.extractall(caminho_pasta_destino)
    
            print(f'{index+1} - Arquivos extraídos de {arquivo}.')

In [5]:
itens_na_pasta = os.listdir(diretorio_cvat)
existe_subpasta = any(os.path.isdir(os.path.join(diretorio_cvat, item)) for item in itens_na_pasta)

if existe_subpasta:
    print("Os arquivos já foram extraídos. Extração ignorada.")
else:
    descompacta_arquivos(diretorio_cvat)

Os arquivos já foram extraídos. Extração ignorada.


## 2. Índice da classe *inside-macrophage* no `obj.names`

In [6]:
# Retorna o indice da classe inside-macrophage a partir do arquivo obj.names
def encontra_indice_classe(diretorio_cvat, classe_alvo):
    for lamina in os.listdir(diretorio_cvat):
        obj_names = os.path.join(diretorio_cvat, lamina, 'obj.names')
        if not os.path.isfile(obj_names): # ignora os .zip e outros arquivos soltos
            continue

        with open(obj_names) as f:
            classes = [linha.strip() for linha in f]

        if classe_alvo in classes:
            return classes.index(classe_alvo) # Retorna o índice da classe alvo

indice_alvo = encontra_indice_classe(diretorio_cvat, classe_alvo)
print(f"Índice da classe '{classe_alvo}': {indice_alvo}")

Índice da classe 'inside-macrophage': 3


## 3. Varrer todas as imagens e manter as que têm anotação *inside-macrophage*

In [7]:
# Retorna True se o .txt de anotação YOLO contiver a classe inside-macrophage.
def tem_classe(txt_path, indice_alvo):
    if not os.path.isfile(txt_path):
        return False
    with open(txt_path, 'r') as f:
        for linha in f:
            linha = linha.strip()
            if not linha:
                continue
            classe_id = int(linha.split()[0])
            if classe_id == indice_alvo:
                return True
    return False


# Retorna DataFrame apenas com imagens que contêm a classe alvo.
def filtra_imagens(diretorio_cvat, subpasta_objetos, extensao, indice_alvo):
    registros = []

    for lamina in sorted(os.listdir(diretorio_cvat)):
        diretorio = os.path.join(diretorio_cvat, lamina, subpasta_objetos)
        if not os.path.isdir(diretorio): # se não for um diretório, pula
            continue

        total = 0
        encontradas = 0
        for arquivo in sorted(os.listdir(diretorio)):
            if not arquivo.endswith(extensao): # se não for a extensão .png, pula
                continue
            total += 1
            nome_base = os.path.splitext(arquivo)[0] # nome_base sem extensão
            txt_path  = os.path.join(diretorio, f"{nome_base}.txt") # caminho do .txt correspondente
            if tem_classe(txt_path, indice_alvo): # se o .txt contiver a classe alvo
                encontradas += 1
                registros.append({
                    'diretorio': diretorio,
                    'imagem': arquivo,
                    'lamina': lamina
                })

        print(f"Lâmina '{lamina}': {encontradas}/{total} imagens com '{classe_alvo}'")

    return pd.DataFrame(registros)


dataset_filtrado = filtra_imagens(diretorio_cvat, subpasta_objetos, extensao, indice_alvo)
print(f"\nTotal de imagens com '{classe_alvo}': {len(dataset_filtrado)}")

Lâmina 'leish_1': 67/67 imagens com 'inside-macrophage'
Lâmina 'leish_2': 46/52 imagens com 'inside-macrophage'
Lâmina 'leish_3': 11/14 imagens com 'inside-macrophage'
Lâmina 'leish_4': 47/47 imagens com 'inside-macrophage'
Lâmina 'leish_5': 127/127 imagens com 'inside-macrophage'
Lâmina 'leish_7': 95/95 imagens com 'inside-macrophage'
Lâmina 'leish_8': 134/134 imagens com 'inside-macrophage'
Lâmina 'leish_9': 72/73 imagens com 'inside-macrophage'

Total de imagens com 'inside-macrophage': 599


## 4. Conferência do resultado

In [8]:
dataset_filtrado

,diretorio,imagem,lamina
0,/mnt/arquivos/Doutorado/dataset_cvat/leish_1/o...,leish_1_0.png,leish_1
1,/mnt/arquivos/Doutorado/dataset_cvat/leish_1/o...,leish_1_1.png,leish_1
2,/mnt/arquivos/Doutorado/dataset_cvat/leish_1/o...,leish_1_10.png,leish_1
3,/mnt/arquivos/Doutorado/dataset_cvat/leish_1/o...,leish_1_11.png,leish_1
4,/mnt/arquivos/Doutorado/dataset_cvat/leish_1/o...,leish_1_12.png,leish_1
...,...,...,...
594,/mnt/arquivos/Doutorado/dataset_cvat/leish_9/o...,Leish_09_72.png,leish_9
595,/mnt/arquivos/Doutorado/dataset_cvat/leish_9/o...,Leish_09_73.png,leish_9
596,/mnt/arquivos/Doutorado/dataset_cvat/leish_9/o...,Leish_09_75.png,leish_9
597,/mnt/arquivos/Doutorado/dataset_cvat/leish_9/o...,Leish_09_76.png,leish_9


In [9]:
print("Distribuição por lâmina para conferência:")
print(dataset_filtrado['lamina'].value_counts())

Distribuição por lâmina para conferência:
lamina
leish_8    134
leish_5    127
leish_7     95
leish_9     72
leish_1     67
leish_4     47
leish_2     46
leish_3     11
Name: count, dtype: int64


## 5. Copiar imagens e `.txt` selecionados para a pasta de saída, organizados por lâmina

In [10]:
# Cria o diretório para o dataset de detecção, se não existir
os.makedirs(diretorio_saida, exist_ok=True)

itens_na_pasta = os.listdir(diretorio_saida)

if itens_na_pasta:
    print("As imagens já foram copiadas. Cópia ignorada.")
else:
    # Copia as imagens e os arquivos .txt correspondentes para o diretório de saída, organizados por lâmina
    for _, row in dataset_filtrado.iterrows():
        nome_base = os.path.splitext(row['imagem'])[0]

        img_origem = os.path.join(row['diretorio'], row['imagem'])
        txt_origem = os.path.join(row['diretorio'], f"{nome_base}.txt")

        pasta_lamina = os.path.join(diretorio_saida, row['lamina'])
        os.makedirs(pasta_lamina, exist_ok=True)

        shutil.copy(img_origem, os.path.join(pasta_lamina, row['imagem']))
        if os.path.isfile(txt_origem):
            shutil.copy(txt_origem, os.path.join(pasta_lamina, f"{nome_base}.txt"))

    print(f"{len(dataset_filtrado)} imagens (+ .txt) copiadas para '{diretorio_saida}'.")

As imagens já foram copiadas. Cópia ignorada.


In [17]:
# Copia o arquivo obj.names para o diretório de saída
caminho_obj_names = os.path.join(diretorio_cvat, dataset_filtrado['lamina'][0], "obj.names")
shutil.copy(caminho_obj_names, diretorio_saida)

'/mnt/arquivos/Doutorado/dataset_macrofagos_infectados/obj.names'